# Spaceship Titanic, XGBoost Baseline

Kaggle's Spaceship Titanic is the modern starter competition (the successor to the original Titanic getting-started one). Binary target `Transported`, mixed feature types, lots of room for feature engineering on the `Cabin` and `PassengerId` strings.

Data: the competition's labeled `train.csv` (8,693 passengers). Kaggle needs a login, so I pull the same file from a public Hugging Face mirror (`Hugo0133/Spaceship-Titanic`). The competition's `test.csv` has no labels, so all evaluation here is a held-out 20% of train plus 5-fold CV.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print("xgboost not installed — falling back to sklearn GradientBoostingClassifier")

DATA = Path("../../data/spaceship_titanic")
DATA.mkdir(parents=True, exist_ok=True)

## Load the data

Columns: `PassengerId, HomePlanet, CryoSleep, Cabin, Destination, Age, VIP, RoomService, FoodCourt, ShoppingMall, Spa, VRDeck, Name, Transported`. Cabin is `<deck>/<num>/<side>`. PassengerId is `<gggg>_<pp>` (group / person-in-group). Most columns have around 2% missing values.

In [2]:
import urllib.request
CSV = DATA / "train.csv"
URL = "https://huggingface.co/datasets/Hugo0133/Spaceship-Titanic/resolve/main/train.csv"
if not CSV.exists():
    urllib.request.urlretrieve(URL, CSV)
df = pd.read_csv(CSV)
print(df.shape, "  target rate:", df.Transported.mean().round(3))
print("missing per column:")
print(df.isna().sum().to_dict())
df.head()

(8693, 14)   target rate: 0.504
missing per column:
{'PassengerId': 0, 'HomePlanet': 201, 'CryoSleep': 217, 'Cabin': 199, 'Destination': 182, 'Age': 179, 'VIP': 203, 'RoomService': 181, 'FoodCourt': 183, 'ShoppingMall': 208, 'Spa': 183, 'VRDeck': 188, 'Name': 200, 'Transported': 0}


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Feature engineering

The bits that actually moved the leaderboard on the real competition:
- Split `Cabin` into deck/num/side.
- Group size from `PassengerId` prefix.
- Total spend = sum of amenity columns; spend==0 is a strong CryoSleep proxy.

In [3]:
def engineer(df):
    df = df.copy()
    df["GroupId"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("GroupId")["GroupId"].transform("count")

    cab = df["Cabin"].fillna("U/0/U").str.split("/", expand=True)
    df["Deck"], df["CabinNum"], df["Side"] = cab[0], cab[1].astype(float, errors="ignore"), cab[2]

    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df[spend_cols] = df[spend_cols].fillna(0)
    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["ZeroSpend"] = (df["TotalSpend"] == 0).astype(int)

    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["HomePlanet"] = df["HomePlanet"].fillna("Earth")

    # cast booleans to int and one-hot the categoricals
    df["CryoSleep"] = df["CryoSleep"].fillna(False).astype(int)
    df["VIP"] = df["VIP"].fillna(False).astype(int)
    df["CabinNum"] = pd.to_numeric(df["CabinNum"], errors="coerce").fillna(-1)
    df["Destination"] = df["Destination"].fillna("TRAPPIST-1e")
    df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Deck", "Side"], drop_first=True)

    drop = ["PassengerId", "Cabin", "GroupId", "Name"]
    return df.drop(columns=[c for c in drop if c in df.columns])

df_fe = engineer(df)
X = df_fe.drop(columns=["Transported"]).values
y = df_fe["Transported"].astype(int).values
print("features:", df_fe.shape[1] - 1)

features: 26


## Train XGBoost (or fall back to sklearn GB)

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=13)

if HAS_XGB:
    model = XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, random_state=13,
        eval_metric="logloss", n_jobs=2,
    )
else:
    model = GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=13)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
auc_cv = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"CV AUC: mean={auc_cv.mean():.3f}  std={auc_cv.std():.3f}")

model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
print(f"test AUC: {roc_auc_score(y_te, proba):.3f}")
print(f"test acc: {accuracy_score(y_te, pred):.3f}")
print(classification_report(y_te, pred, target_names=["not_transported", "transported"]))

CV AUC: mean=0.905  std=0.010


test AUC: 0.893
test acc: 0.790
                 precision    recall  f1-score   support

not_transported       0.80      0.77      0.79       863
    transported       0.78      0.81      0.79       876

       accuracy                           0.79      1739
      macro avg       0.79      0.79      0.79      1739
   weighted avg       0.79      0.79      0.79      1739



## Notes

On the real competition train.csv, the held-out 20% (1,739 passengers) comes out at **79.0% accuracy and ROC-AUC 0.893**, with 5-fold CV AUC of 0.905 on the training part. That sits right in the 0.79-0.82 accuracy band that XGBoost with decent feature engineering usually gets on the public leaderboard, so the pipeline behaves the way it should.

What moved things:
- The `engineer()` function: deck/num/side from `Cabin`, group size from `PassengerId`, total spend and a zero-spend flag (a strong CryoSleep proxy).
- The XGBoost hyperparameters are a reasonable starting band but untuned. Optuna or a coarse grid is the obvious next step.

Things I'd still do: impute CryoSleep from zero spend instead of filling with False, use the group ID to share information between passengers traveling together, and check that group members don't land on both sides of the split (they currently can, which may make the holdout number slightly optimistic).